# Hyperparameter tuning

The random forest chosen in [`06_model_selection.ipynb`](06_model_selection.ipynb) was trained with default hyperparameters. We now search for better ones with **randomized search** evaluated by cross-validation, in two iterations: a broad one to find the promising regions, and a narrower one around the best results (see [Hyperparameter search](../notes/hyperparameter_search.ipynb) for the comparison with grid search and the meaning of each hyperparameter). Since the preprocessing is part of the pipeline, the number of geographic clusters is tuned at the same time.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from utils.load_california import load_housing_data
from utils.housing_preprocessing import get_preprocessing_pipeline

X_train, X_test, y_train, y_test = load_housing_data()

full_pipeline = Pipeline([
    ("preprocessing", get_preprocessing_pipeline(n_clusters=10)),  # n_clusters will be overridden by the search
    ("random_forest", RandomForestRegressor(random_state=42)),
])

The search parallelizes over candidates with `n_jobs=-1`, so the forest itself keeps its default single-process training.

## First iteration: broad ranges

40 random combinations, each evaluated with 5-fold cross-validation (200 trainings). `%%time` reports how long the cell took.

In [ ]:
%%time

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "preprocessing__geo__n_clusters": randint(low=3, high=200),
    "random_forest__n_estimators": randint(100, 500),  # number of trees
    "random_forest__max_depth": randint(10, 110),  # maximum depth of each tree
    "random_forest__min_samples_split": randint(2, 20),
    "random_forest__min_samples_leaf": randint(1, 20),
    "random_forest__max_features": ["sqrt", "log2", None],  # features considered at each split
}

rnd_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=40,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)

_ = rnd_search.fit(X_train, y_train)

CPU times: user 26.5 s, sys: 413 ms, total: 27 s
Wall time: 13min 9s


In [ ]:
def search_results(search, param_names):
    """Tested combinations sorted by mean validation RMSE (best first)."""
    cv_res = pd.DataFrame(search.cv_results_)
    cv_res = cv_res[[f"param_{name}" for name in param_names] + ["mean_test_score"]]
    cv_res.columns = [name.split("__")[-1] for name in param_names] + ["mean_val_rmse"]
    cv_res["mean_val_rmse"] = -cv_res["mean_val_rmse"].round().astype(np.int64)
    return cv_res.sort_values("mean_val_rmse").fillna("None")  # max_features=None would show as NaN

search_results(rnd_search, list(param_distributions)).head(10)

,n_clusters,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,mean_val_rmse
34,56,225,96,2,1,sqrt,41574
3,132,413,73,13,1,sqrt,42508
25,145,259,52,14,4,sqrt,43156
17,165,330,44,11,5,sqrt,43172
21,103,104,56,2,3,log2,43476
27,64,143,66,13,6,sqrt,43488
7,75,341,48,15,4,log2,43872
35,132,302,62,5,6,log2,44616
20,106,326,95,2,3,None,45307
2,119,393,109,13,2,None,45505


All the top results use `max_features="sqrt"` or `"log2"`, with `"sqrt"` in the first positions, so it is fixed. The other hyperparameters are narrowed to the intervals where the best results concentrate: 55–150 clusters, 200–300 trees, depth 44–97, `min_samples_split` up to 14 and `min_samples_leaf` up to 5.

## Second iteration: narrowed ranges

In [ ]:
%%time

full_pipeline.set_params(random_forest__max_features="sqrt")  # converged in the first iteration

param_distributions = {
    "preprocessing__geo__n_clusters": randint(low=55, high=150),
    "random_forest__n_estimators": randint(200, 300),
    "random_forest__max_depth": randint(44, 97),
    "random_forest__min_samples_split": randint(2, 14),
    "random_forest__min_samples_leaf": randint(1, 5),
}

rnd_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=40,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)

_ = rnd_search.fit(X_train, y_train)

/home/avidaldo/TRABAJO/wip-clase/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


CPU times: user 31.1 s, sys: 254 ms, total: 31.4 s
Wall time: 2min 26s


`joblib` may warn that *a worker stopped while some jobs were given to the executor*: worker processes that sat idle at the end of the first search are recycled when the second one starts, and the jobs assigned to them are simply re-dispatched. It does not affect the results.

In [ ]:
search_results(rnd_search, list(param_distributions)).head(10)

,n_clusters,n_estimators,max_depth,min_samples_split,min_samples_leaf,mean_val_rmse
6,76,290,87,2,1,41604
14,89,249,57,5,1,41678
27,116,243,67,4,1,41747
9,118,206,46,4,1,41846
8,134,254,58,4,2,41848
3,78,201,46,6,2,41858
17,98,213,77,5,2,41860
19,141,252,83,3,1,41864
33,58,214,73,8,1,41920
15,58,253,45,7,2,41923


## Best model

`best_estimator_` is the pipeline with the best combination, already refitted on the whole training set.

In [ ]:
rnd_search.best_params_

{'preprocessing__geo__n_clusters': 76,
 'random_forest__max_depth': 87,
 'random_forest__min_samples_leaf': 1,
 'random_forest__min_samples_split': 2,
 'random_forest__n_estimators': 290}

In [ ]:
print(f"Best cross-validation RMSE: {-rnd_search.best_score_:,.0f}")

Best cross-validation RMSE: 41,604


## Final evaluation on the test set

The test set is used now for the first and only time, to estimate how the final model will perform on data it has never seen (see [Data leakage](../notes/data_leakage.ipynb)).

In [ ]:
from sklearn.metrics import root_mean_squared_error

final_model = rnd_search.best_estimator_
final_predictions = final_model.predict(X_test)

print(f"Test RMSE: {root_mean_squared_error(y_test, final_predictions):,.0f}")

Test RMSE: 39,217


The test RMSE is close to the cross-validation estimate, which is what we expect when the test set has not been used along the way. The hyperparameters found here (number of clusters and the forest's settings) are reused as the baseline in [`08_neural_network.ipynb`](08_neural_network.ipynb).